from pydantic_ai.providers.mistral import MistralProviderfrom pydantic_ai.models.mistral import MistralModel### 1. <u>_PYDANTIC AI_</U>
* _Pydantic AI is a `Python framework for building GenAI agents`._
 * _It provides `strong typing`, `validation`, `structured outputs`, `tools`, `dependency injection`, `streaming`, and `model portability`._
___
#### 1.1 <u>_Advantage of Pydantic AI_</u>
* _A big advantage is that the framework is `vendor-agnostic at the agent level`._
* The same agent can run against different _providers by changing the model object or model string_, _rather than rewriting your whole orchestration layer_
___
#### 1.2 <u>_What is an AI Agent_</u>
```text
            User Request                      A AGENT GENERALLY CONSIST OF:
            |                                    1. Instructions: what role it plays
            v                                    2. Model: which LLM it uses
        +------------------+                     3. Tools: functions/APIs/databases it can call
        |  Agent Brain     |                     4. Memory/State: history, saved facts, external store
        |  (LLM + rules)   |                     5. Output schema: structured response contract
        +------------------+                     6. Planning/Control: how it chooses steps
            | decides                            7. Observability: logs/traces/usage
            +--------------------+
            |                    |
            v                    v
        Use Tool?            Answer Directly
            |                    |
            v                    v
        Call function/API     Return response
            |
            v
        Use result + continue reasoning
```

### 2. <U>_Agent in Pydantic AI_</U>
* _Agent is the main component of the Pydantic AI._
```text
    +------------------------------------------------------+
    |                     Agent                            |
    |------------------------------------------------------|
    | Instructions                                         |
    | Model / Provider                                     |
    | Tools / Toolsets                                     |
    | Output Type (Pydantic schema)                        |
    | Dependencies                                         |
    | Model Settings                                       |
    | Capabilities                                         |
    +------------------------------------------------------+
```

```python

```

#### 2.1 Model & Provider
* _Model = wrapper for a family/API style_
* _Provider = authentication and endpoint handling (API_KEY)_

In [ ]:
from pydantic_ai import Agent
from  pydantic_ai.models.mistral import MistralModel
from  pydantic_ai.providers.mistral import MistralProvider

mistral_agent = Agent(
    model = MistralModel(
        model_name= "A_VALID_MISTRAL_MODEL_NAME",                       # Model
        provider= MistralProvider(api_key= "VALID MISTRAL API KEY")     # Provider
    )
)

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider

from config.settings import settings
gemini_agent = Agent(
    model= GoogleModel(
        model_name= settings.GEMINI_MODEL,
        provider= GoogleProvider(api_key=settings.GEMINI_API_KEY)
    ),
    instructions= "Be consise with the answers.",
)

response = await gemini_agent.run("Who is modi ?")
print(response)


#### 2.2 <u>_system_prompt vs instructions_</u>
```text
        1. instructions
           => What this agent should do right now
           => Current agent behavior / style / task guidance
           => Use cases: task style, explanation depth, tone, formatting, audience level
        2. system_prompt:
           =>  System-level steering that can stay in message history
           =>  Hard guardrails / persistent system rules
           =>  Use Cases : security, compliance, role boundaries, non-negotiable rules

```

##### 2.2.1 <u>_Use with system_prompt_</u>
* _Here you are setting a more **“system-like” rule**_.
* _If this **prompt is in the message history** and you continue the conversation with that history_.
* _it **can continue to influence future runs in a way that instructions are not intended to**_.

In [ ]:
from pydantic_ai import Agent
from  pydantic_ai.models.mistral import MistralModel
from  pydantic_ai.providers.mistral import MistralProvider
from config.settings import settings
mistral_agent1 = Agent(
    model = MistralModel(
        model_name= settings.MISTRAL_MODEL,                                 # Model
        provider= MistralProvider(api_key= settings.MISTRAL_API_KEY)        # Provider

    ),
    system_prompt= """
    You are a bot for the company BridgeLabz.
    Help line number : 111-1111-1111
    """
)
response = await mistral_agent1.run("What is the help line number ?")
print(response)

##### 2.2.2 <u>_Use with Instruction_</u>
* _This agent behaves like a Python tutor for this run._
* _If **later you reuse message history with another agent**, the **earlier instructions are not reused from past messages**_
 * _The new agent’s own instructions are what matter_

In [ ]:
from pydantic_ai import Agent
from  pydantic_ai.models.mistral import MistralModel
from  pydantic_ai.providers.mistral import MistralProvider
from config.settings import settings
mistral_agent2 = Agent(
    model = MistralModel(
        model_name= settings.MISTRAL_MODEL,                                 # Model
        provider= MistralProvider(api_key= settings.MISTRAL_API_KEY)        # Provider

    ),
    instructions= """
    You are a Python tutor.
    Explain concepts step by step.
    Use simple examples first.
    """
)
response2 = await mistral_agent2.run("What is Inheritance ?")
print(response2)

##### 2.2.3 <u>_System prompt vs Instructions_</u>
* _mistral_agent1 : uses system_prompt_
* _mistral_agent2 : uses instructions_
* <u>_If mistral_agent1 has already runed before_</u>:
    * => _Any agent running after that will have the system_prompt context info of the Previous Agent._
* _`note`:=> If strict instruction is defined for that it will not give the info regarding previous agent system_prompt context(e.g, agent2)_
* <u>_e.g,_</u>
  * => _mistral_agent3 will have the info regarding bridgelabz help-line num. even though it's context is not provided with in the agent2_


In [ ]:
mistral_agent3 = Agent(
    model = MistralModel(
        model_name= settings.MISTRAL_MODEL,                                 # Model
        provider= MistralProvider(api_key= settings.MISTRAL_API_KEY)        # Provider
    ),
    system_prompt="you are a simple chat bot"
)
response3 = await mistral_agent3.run("What is the help line number for bridgelabz ?")
print(response3)

___

#### 2.3 <u>_Dynamic Instructions_</u>
* Pydantic AI also supports dynamic instructions using @agent.instructions
* This Instructions arealways reevaluated at runtime.